In [1]:
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup

all_extracted_data = []
max_page = 50
# Loop over the pages directly
for page in range(1, max_page):
    print(f"--- Scraping Page {page} ---")
    
    try:
        url = 'https://www.propertyfinder.eg/en/buy/properties-for-sale.html'
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        }
        
        response = requests.get(url, headers=headers, params={'page': page}, timeout=10)
        response.encoding = 'utf-8'
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            tag = soup.find("meta", {"itemprop": "numberOfItems"})
            
            if tag != None:
                if int(tag['content']) == 0:
                    print('No More Properties')
                    print("Stopping signal received. Finishing up...")
                    break  # Stop the loop
                
                script_tag = soup.find('script', id='__NEXT_DATA__')
                
                if script_tag != None:
                    data = json.loads(script_tag.string)
                    listings = data.get('props', {}).get('pageProps', {}).get('searchResult', {}).get('listings', [])
                    
                    if len(listings) == 0:
                        print("No more listings found.")
                        print("Stopping signal received. Finishing up...")
                        break  # Stop the loop
                    
                    page_data_count = 0
                    
                    for item in listings:
                        prop = item.get('property', {})
                        if prop:
                            info = {}
                            info['title'] = prop.get('title')
                            info['price'] = prop.get('price', {}).get('value')
                            info['location'] = prop.get('location', {}).get('full_name')
                            info['area'] = prop.get('size', {}).get('value')
                            info['property_type'] = prop.get('property_type')
                            info['bedrooms'] = prop.get('bedrooms')
                            info['bathrooms'] = prop.get('bathrooms')
                            
                            all_extracted_data.append(info)
                            page_data_count += 1
                            
                    print(f"Done page {page}. Items: {page_data_count}")
                    
                else:
                    print(f"Could not find __NEXT_DATA__ on page {page}")
            else:
                print("Meta tag not found")
        else:
            print(f"Failed to fetch page {page}. Status code: {response.status_code}")
            
    except Exception as e:
        print(f"Error on page {page}: {str(e)}")

# Save to pandas and export
df = pd.DataFrame(all_extracted_data)
df.to_csv('property_listings.csv', index=False, encoding='utf-8-sig')

print(f"\nSuccess! Saved {len(all_extracted_data)} properties to property_listings.csv")

--- Scraping Page 1 ---
Done page 1. Items: 25
--- Scraping Page 2 ---
Done page 2. Items: 25
--- Scraping Page 3 ---
Done page 3. Items: 25
--- Scraping Page 4 ---
Done page 4. Items: 25
--- Scraping Page 5 ---
Done page 5. Items: 25
--- Scraping Page 6 ---
Done page 6. Items: 25
--- Scraping Page 7 ---
Done page 7. Items: 25
--- Scraping Page 8 ---
Done page 8. Items: 25
--- Scraping Page 9 ---
Done page 9. Items: 25
--- Scraping Page 10 ---
Done page 10. Items: 25

Success! Saved 250 properties to property_listings.csv
